# M1 — zero-shot topic re-tagging (MDCAT rows only)

`tag_topics.py`'s keyword heuristic put 67% of the 1777 MDCAT-derived
questions into a generic "General &lt;Subject&gt;" bucket, which is the main
reason M1 (Knowledge Gap Classifier) only reached F1 (micro) = 0.4691 after
attempt 3 (rare-class merge + class weighting) against the >=0.85 target —
see `notebooks/experiment_log.md`.

This notebook re-tags those same 1777 rows with `facebook/bart-large-mnli`
zero-shot classification (reads each question and scores it against the
real topic names for its subject, instead of substring-matching keywords)
and writes the result as a **new column**, never overwriting the current
`topic` column, so you can compare before accepting.

**Logical Reasoning (216 rows) is skipped** — the doc deliberately treats
it as one bucket (not worth subdividing a 216-question set), so there's
nothing for zero-shot to improve there.


## Setup — mount Drive, confirm GPU, install deps

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
# Expect to see "Tesla T4" — if this errors or shows no GPU, go to
# Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU -> Save, then re-run.


Mounted at /content/drive
Thu Oct  8 16:46:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+---------------------

In [2]:
DATA = '/content/drive/MyDrive/mentora_data'


In [3]:
!pip install -q torch --index-url https://download.pytorch.org/whl/cu121 --break-system-packages
!pip install -q transformers accelerate pandas --break-system-packages


## Load data + candidate topic lists per subject

In [4]:
import pandas as pd

df = pd.read_csv(f'{DATA}/processed/m1/question_bank_final.csv')
print(df.shape)
print(df['topic'].value_counts())

# Same topic names as datasets/m1_gap_classifier/tag_topics.py's TAXONOMY,
# plus an explicit "General <Subject>" option so the model can genuinely
# pick it (rather than being forced into a specific-but-wrong topic).
CANDIDATES = {
    'Biology': ['Cell Biology', 'Genetics', 'Physiology', 'Botany', 'Zoology',
                'Ecology', 'Evolution', 'Biochemistry', 'Microbiology', 'General Biology'],
    'Chemistry': ['Organic Chemistry', 'Inorganic Chemistry', 'Physical Chemistry',
                  'Biochemistry', 'General Chemistry'],
    'Physics': ['Mechanics', 'Electricity & Magnetism', 'Waves & Optics',
                'Thermodynamics', 'Modern Physics', 'General Physics'],
    'English': ['Grammar', 'Vocabulary', 'Reading Comprehension',
                'Writing & Punctuation', 'General English'],
}
# Logical Reasoning intentionally excluded — single bucket by design, see above.

mdcat_mask = df['subject'].isin(CANDIDATES.keys())
print(f"\n{mdcat_mask.sum()} rows to re-tag across {list(CANDIDATES.keys())}")


(2155, 9)
topic
General Chemistry                    381
General Biology                      357
General Physics                      233
General Logical Reasoning            216
General English                      129
Algebra                              124
Advanced Math                        107
Problem-Solving and Data Analysis     97
Mechanics                             73
Electricity & Magnetism               55
Genetics                              51
Geometry and Trigonometry             50
Physical Chemistry                    49
Waves & Optics                        48
Physiology                            44
Biochemistry                          32
Organic Chemistry                     31
Modern Physics                        27
Inorganic Chemistry                   26
Botany                                25
Name: count, dtype: int64

1561 rows to re-tag across ['Biology', 'Chemistry', 'Physics', 'English']


## Zero-shot classify each subject's rows against its own candidate list

In [5]:
from transformers import pipeline
import torch

device = 0 if torch.cuda.is_available() else -1
classifier = pipeline('zero-shot-classification', model='facebook/bart-large-mnli', device=device)

df['zeroshot_topic'] = df['topic']
df['zeroshot_confidence'] = 1.0

for subject, candidates in CANDIDATES.items():
    idx = df.index[df['subject'] == subject]
    texts = df.loc[idx, 'question_text'].astype(str).tolist()
    print(f"{subject}: classifying {len(texts)} rows against {candidates}")
    results = classifier(texts, candidate_labels=candidates, batch_size=16)
    if isinstance(results, dict):   # single-text edge case returns a dict, not a list
        results = [results]
    for i, r in zip(idx, results):
        df.at[i, 'zeroshot_topic'] = r['labels'][0]
        df.at[i, 'zeroshot_confidence'] = r['scores'][0]

print("\nNew topic distribution (MDCAT rows, zero-shot):")
print(df.loc[mdcat_mask, 'zeroshot_topic'].value_counts())


config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Biology: classifying 503 rows against ['Cell Biology', 'Genetics', 'Physiology', 'Botany', 'Zoology', 'Ecology', 'Evolution', 'Biochemistry', 'Microbiology', 'General Biology']
Chemistry: classifying 493 rows against ['Organic Chemistry', 'Inorganic Chemistry', 'Physical Chemistry', 'Biochemistry', 'General Chemistry']
Physics: classifying 436 rows against ['Mechanics', 'Electricity & Magnetism', 'Waves & Optics', 'Thermodynamics', 'Modern Physics', 'General Physics']
English: classifying 129 rows against ['Grammar', 'Vocabulary', 'Reading Comprehension', 'Writing & Punctuation', 'General English']

New topic distribution (MDCAT rows, zero-shot):
zeroshot_topic
Mechanics                  197
Physiology                 150
Biochemistry               149
Physical Chemistry         138
General Chemistry          119
Microbiology                85
Organic Chemistry           80
Waves & Optics              79
Electricity & Magnetism     61
General English             58
Genetics            

## Spot-check before accepting

Compare old heuristic topic vs new zero-shot topic + confidence on a random
sample, and look specifically at how many of the old "General &lt;Subject&gt;"
rows got a more specific label now. Read through the sample below before
trusting the new column — zero-shot is much better than keyword matching
but still not ground truth.

In [6]:
sample = df.loc[mdcat_mask, ['subject', 'question_text', 'topic', 'zeroshot_topic', 'zeroshot_confidence']]
print(sample.sample(25, random_state=42).to_string())

was_general = df['topic'].str.startswith('General', na=False) & mdcat_mask
now_specific = was_general & ~df['zeroshot_topic'].str.startswith('General', na=False)
print(f"\n{was_general.sum()} rows were 'General <Subject>' before; "
      f"{now_specific.sum()} of those got a specific zero-shot topic now "
      f"({now_specific.sum() / max(was_general.sum(), 1):.0%})")

print("\nConfidence distribution:")
print(df.loc[mdcat_mask, 'zeroshot_confidence'].describe())


        subject                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    question_text                    topic      zeroshot_topic  zeroshot_confidence
1638  Chemistry                                                                                      

## Save

Writes `question_bank_final_zeroshot.csv` alongside the original — it does
**not** overwrite `question_bank_final.csv`. After reviewing the spot-check
above:
1. Download this new CSV from Drive.
2. Locally, run `datasets/m1_gap_classifier/merge_rare_topics.py` against it
   as a safety net (zero-shot may still leave some topic under the 25-example
   floor).
3. Replace your local `mentora_data/processed/m1/question_bank_final.csv`
   with the result, re-upload to Drive, delete
   `mentora_models/model1_gap_classifier/checkpoint-*` (label set will have
   changed again), and re-run `train_m1.ipynb` for attempt 4.

In [7]:
out = df.drop(columns=['topic']).rename(columns={'zeroshot_topic': 'topic'})
out = out.drop(columns=['zeroshot_confidence'])
out_path = f'{DATA}/processed/m1/question_bank_final_zeroshot.csv'
out.to_csv(out_path, index=False)
print(f"Saved to {out_path}")

# Also keep the confidence scores separately for your own reference/QA.
df[['question_text', 'subject', 'topic', 'zeroshot_topic', 'zeroshot_confidence']].to_csv(
    f'{DATA}/processed/m1/zeroshot_retag_review.csv', index=False
)
print(f"Saved review file to {DATA}/processed/m1/zeroshot_retag_review.csv")


Saved to /content/drive/MyDrive/mentora_data/processed/m1/question_bank_final_zeroshot.csv
Saved review file to /content/drive/MyDrive/mentora_data/processed/m1/zeroshot_retag_review.csv
